# softmax简洁实现


In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

## 初始化模型参数
`Sequential`类将多个层串联在一起。当给定输入数据时，他的实例将数据传入第一层，然后将第一层输出作为第二层输入......

在 PyTorch 中全连接层在`Linear`类中定义 

在 softmax 中输出层是一个全连接层，只需在`Sequential`中添加一个带有 10 个输出的全连接层。


In [3]:
# PyTorch不会隐式地调整输入的形状。因此，
# 我们在线性层前定义了展平层（flatten），来调整网络输入的形状 28*28
net = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))

def init_weights(m):
    if type(m) == nn.Linear:    #判断当前模块是不是全连接层
        nn.init.normal_(m.weight, std=0.01) #第一个参数是要被初始化的张量

net.apply(init_weights);

## softmax处理上溢下溢
上溢：分子分母同时减去 max
$$
\begin{aligned}
\hat y_j & =  \frac{\exp(o_j - \max(o_k))\exp(\max(o_k))}{\sum_k \exp(o_k - \max(o_k))\exp(\max(o_k))} \\
& = \frac{\exp(o_j - \max(o_k))}{\sum_k \exp(o_k - \max(o_k))}.
\end{aligned}
$$

下溢：计算交叉熵要取对数，在 softmax 中与其结合，避免反向传播时一屏幕的`nan`
$$
\begin{aligned}
\log{(\hat y_j)} & = \log\left( \frac{\exp(o_j - \max(o_k))}{\sum_k \exp(o_k - \max(o_k))}\right) \\
& = \log{(\exp(o_j - \max(o_k)))}-\log{\left( \sum_k \exp(o_k - \max(o_k)) \right)} \\
& = o_j - \max(o_k) -\log{\left( \sum_k \exp(o_k - \max(o_k)) \right)}.
\end{aligned}
$$



In [ ]:
loss = nn.CrossEntropyLoss(reduction='none')

trainer = torch.optim.SGD(net.parameters(), lr=0.1)

num_epochs = 10
d2l.train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)